In [1]:
import os
import math
import json
import numpy as np
import pandas as pd

from pyspark.sql.functions import pandas_udf, hour, log, cos, sin, radians, col, lit

In [2]:
@pandas_udf("double")
def haversine(lon1:pd.Series,lat1:pd.Series,lon2:pd.Series,lat2:pd.Series)->pd.Series:
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat*0.5)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon*0.5)**2
    return 6378137.0 * 2.0 * np.arcsin(np.sqrt(a))

In [3]:
schema = ",".join([
    "id string",
    "vendor_id string",
    "pickup_datetime timestamp",
    "dropoff_datetime timestamp",
    "passenger_count string",
    "pickup_longitude double",
    "pickup_latitude double",
    "dropoff_longitude double",
    "dropoff_latitude double",
    "store_and_fwd_flag string",
    "trip_duration double"
])

load_path = os.path.expanduser(os.path.join("~","Kaggle","nyc-taxi-trip-duration","train.csv"))

xmin,ymin,xmax,ymax = (-74.2555913638106, 40.496115395209344, -73.70000906387119, 40.91553277600007)
dmin,dmax = (120.0, 6000.0)

where = " and ".join([
    f"pickup_longitude between {xmin} and {xmax}",
    f"pickup_latitude between {ymin} and {ymax}",
    f"dropoff_longitude between {xmin} and {xmax}",
    f"dropoff_latitude between {ymin} and {ymax}",
    f"trip_duration between {dmin} and {dmax}"
])

df = spark\
    .read\
    .format("csv")\
    .option("header", True)\
    .option("parserLib", "univocity")\
    .option("mode", "permissive")\
    .schema(schema)\
    .load(load_path)\
    .where(where)\
    .withColumn("pickup_hour", hour("pickup_datetime"))\
    .withColumn("hour_rad", radians(360.0*col("pickup_hour")/23.0))\
    .withColumn("hour_cos", cos("hour_rad"))\
    .withColumn("hour_sin", sin("hour_rad"))\
    .withColumn("log_duration", log("trip_duration"))\
    .withColumnRenamed("pickup_longitude", "plon")\
    .withColumnRenamed("pickup_latitude", "plat")\
    .withColumnRenamed("dropoff_longitude", "dlon")\
    .withColumnRenamed("dropoff_latitude", "dlat")\
    .withColumn("meters", haversine("plon","plat","dlon","dlat"))\
    .withColumn("mps", col("meters")/col("trip_duration"))\
    .filter("mps between 0.5 and 10.0")\
    .drop("id",
          "mps",
          "vendor_id",
          "passenger_count",
          "store_and_fwd_flag",
          "pickup_hour",
          "hour_rad",
          "trip_duration",
          "pickup_datetime",
          "dropoff_datetime")

df.createOrReplaceTempView("v0")

In [4]:
df.printSchema()

root
 |-- plon: double (nullable = true)
 |-- plat: double (nullable = true)
 |-- dlon: double (nullable = true)
 |-- dlat: double (nullable = true)
 |-- hour_cos: double (nullable = true)
 |-- hour_sin: double (nullable = true)
 |-- log_duration: double (nullable = true)
 |-- meters: double (nullable = true)



In [5]:
rows = sql("""select
min(meters) min_meters,max(meters) max_meters
from v0
""")\
    .show()

+------------------+-----------------+
|        min_meters|       max_meters|
+------------------+-----------------+
|61.083860959640596|42315.01044303137|
+------------------+-----------------+



In [6]:
mmin = 60.0
mmax = 43000.0

dmin = math.log(dmin)
dmax = math.log(dmax)

params = {
    "xmin": xmin,
    "ymin": ymin,
    "xmax": xmax,
    "ymax": ymax,
    "meters_min": mmin,
    "meters_max": mmax,
    "log_dur_min": dmin,
    "log_dur_max": dmax
}

with open(os.path.expanduser(os.path.join("~","trips.json")),"w") as f:
     json.dump(params,f,indent=2)

In [7]:
xdel = xmax - xmin
ydel = ymax - ymin
mdel = mmax - mmin
ddel = dmax - dmin

Normalize the data.

In [8]:
df = sql(f"""select
(plon - {xmin})/{xdel} plon,
(plat - {ymin})/{ydel} plat,
(dlon - {xmin})/{xdel} dlon,
(dlat - {ymin})/{ydel} dlat,
(meters - {mmin})/{mdel} meters,
(hour_cos + 1.0)/2.0 hour_cos,
(hour_sin + 1.0)/2.0 hour_sin,
(log_duration - {dmin})/{ddel} log_duration
from v0
""")

In [9]:
df.write.parquet(os.path.expanduser(os.path.join("~","trips.par")),mode="overwrite")